In [ ]:
import sys, os, warnings, subprocess
subprocess.run(["pip", "install", ".", "--quiet"], check=True,
               cwd="/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales")
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import scipy as sp
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

%matplotlib inline
plt.rcParams.update({
    'font.size':         8,
    'figure.dpi':        300,
    'svg.fonttype':      'none',
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Roboto'],
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

sys.path.extend([r'/mnt/storage_ssd_raid/research_projects/snaplab_tools'])

from src.notebook import load_notebook_data
from src.timescales import (create_ints_dataframe_custom_grouping, convert_data_dict,
                             average_data_dict_converted, compute_deflections)
from src.plotting import register_custom_colormaps, show_colormaps
from snaplab_tools.gams import detect_change_point, partial_r2_smooth
from src.gams import (fit_all_panels, apply_region_mask, get_series_colors, get_draw_order,
                      style_ax, compute_deriv_sa_corr, bootstrap_gam_maps,
                      add_sa_decile_colorbar, plot_deriv_sa_corr_with_ci)

# Load data

In [ ]:
dataset       = 'HCP-D' # 'HCP-D' 'PNC'
which_modality = 'FMRI'
n_regions = 400

if dataset == 'HCP-D':
    # output_type   = 'timescale_auc_zero_crossing'
    output_type   = 'timescale_auc_zero_crossing_bp-0.01-0.08hz'
elif dataset == 'PNC':
    output_type   = 'timescale_auc_zero_crossing'
    # output_type   = 'timescale_zero_crossing'
    # output_type   = 'timescale_zero_crossing_robust_nc2'
    # output_type   = 'timescale_zero_crossing_robust_nc3'

notebook_data = load_notebook_data(dataset=dataset, which_modality=which_modality,
                                   output_type=output_type, use_resids=True,
                                   load_subject_data=False, n_regions=n_regions)

In [ ]:
timescales_processor = notebook_data['timescales_processor']
data_dict            = notebook_data['data_dict']
results_dir          = notebook_data['results_dir']
df_neuro             = notebook_data['supporting_data']['df_neuro']

sa_axis = df_neuro['SA_Axis'].values

nas_scratch_dir = f'/mnt/lindenmp_nas/lindenmp/work/figure_scratch/intrinsic_neural_timescales/{dataset}'
os.makedirs(nas_scratch_dir, exist_ok=True)

In [ ]:
sa_axis.shape

# Prepare data

In [ ]:
timescales_processor.match_registry_to_behavior()
df_behavior = timescales_processor.df_behavior_matched.copy()

if dataset == 'HCP-D':
    rest_scan = 'task-Rest'
    # task_scan = 'tfMRIEMOTIONPA'
    task_scan = 'tfMRICARITPA'
    # task_scan = 'tfMRIGUESSINGPA'
elif dataset == 'PNC':
    rest_scan = 'rest'
    task_scan = 'frac2back'

data_dict_tsnr      = notebook_data['data_dict_tsnr']
data_dict_converted = convert_data_dict(data_dict, timescales_processor.scan_registry)
data_dict_centiles  = convert_data_dict(data_dict, timescales_processor.scan_registry,
                                        brain_map=sa_axis)
data_dict_converted = average_data_dict_converted(data_dict_converted, average_all_rest=True)
data_dict_centiles  = average_data_dict_converted(data_dict_centiles,  average_all_rest=True)
df_tsnr             = create_ints_dataframe_custom_grouping(data_dict_tsnr, average_all_rest=True,
                                                            n_regions=n_regions)

# Fit GAMs

In [ ]:
df_behavior

In [ ]:
# ── GAM + bootstrap configuration ────────────────────────────────
# Fitting recipe (switch fit_recipe). Both are kept available:
#   'historical' -> n_splines=4, lam=0.1 FIXED (the original recipe; writes the original cache file)
#   'gcv'        -> n_splines=8, lam selected per region by GCV (flexible; separate cache file)
# GCV self-regulates by SNR: the high-SNR decile curves gain real curvature (2nd-derivative
# structure) while individual regions self-limit to ~2-3 effective DOF, so the regional analysis
# stays honest. lam is selected once on the full sample and frozen for the bootstrap.
fit_recipe = 'gcv'
if fit_recipe == 'historical':
    n_splines = 4
    lam = 0.1
    recipe_tag = ''
elif fit_recipe == 'gcv':
    n_splines = 5
    lam = 'gcv'
    recipe_tag = f'_ns{n_splines}-gcv'
else:
    raise ValueError(f"fit_recipe must be 'historical' or 'gcv'; got {fit_recipe!r}")

absolute            = True
n_boot              = 500      # bootstrap iterations (derivative × S-A CI + fig5 stage maps)
boot_seed           = 42        # resampling seed (shared by all bootstrap reductions)
N_JOBS              = -1        # joblib workers for the bootstrap (-1 = all cores)

# Data-driven stage boundary: 'deriv1' (rate reorganization) is stable across fitting recipes;
# deriv2 (acceleration) is near-degenerate at the regional level (its boundary drifts with
# flexibility), so it is only REPORTED as a robustness CI, not used to define the stage maps.
stage_boundary_signal   = 'deriv1'       # drives the exported stage maps ('deriv1'|'deriv2'|'pred')
boundary_report_signals = ('deriv2', 'pred')    # extra signals to report a boundary CI for (maps unaffected)

# Number of change points to detect for the boundary annotation (the figure draws this many
# vertical age lines). n_bkps=1 -> the single stage boundary (two stages). n_bkps>1 uses ruptures
# (exploratory; the smooth GAM signals tend to over-segment) and does NOT change the two-stage
# stage maps exported for fig5 — those still use the single-boundary detection.
n_bkps = 1

# The GAM fits + bootstrap are slow, so their outputs are cached to gam_cache_file and reloaded on
# re-runs. Set FORCE_RERUN = True to recompute from scratch and overwrite the cache.
# NOTE: regenerate the cache (True) after changing the fitting recipe or stage-boundary definition —
# the exported stage maps and fig5's inputs depend on them. recipe_tag keeps recipes in separate files.
FORCE_RERUN    = False
gam_out_dir    = os.path.join(timescales_processor.output_dir, 'results', dataset, 'gam')
os.makedirs(gam_out_dir, exist_ok=True)
gam_cache_file = os.path.join(gam_out_dir, f'{dataset}_{n_regions}_{task_scan}{recipe_tag}_regional_gam_results.npy')

In [ ]:
# Load the cached GAM + bootstrap outputs if present, else compute them from scratch and save.
if (not FORCE_RERUN) and os.path.exists(gam_cache_file):
    save_dict = np.load(gam_cache_file, allow_pickle=True).item()
    print(f'Loaded cached GAM + bootstrap outputs <- {gam_cache_file}')
    print('  (set FORCE_RERUN = True in the config cell above to recompute and overwrite)')
else:
    # ===== Region- and decile-level data =====
    rest_full = data_dict_converted[rest_scan]
    try:
        task_full = data_dict_converted[task_scan]
    except (KeyError, TypeError):
        task_full = data_dict_converted[rest_scan]
    n_subs = task_full.shape[0]
    n_regions = task_full.shape[1]

    # Covariate data (age smooth + nuisance: sex, meanFD, and each subject's mean rest INT)
    age = df_behavior['age'].values
    if dataset == 'HCP-D':
        nuisance = np.concat((df_behavior[['sex', 'meanFD']].values,
                              np.nanmean(rest_full, axis=1)[:, np.newaxis]), axis=1)
    elif dataset == 'PNC':
        nuisance = np.concat((df_behavior[['sex', 'meanFD_rest']].values,
                              np.nanmean(rest_full, axis=1)[:, np.newaxis]), axis=1)
    else:
        nuisance = None
    age_grid = np.linspace(age.min(), age.max(), 200)

    deflection_full = compute_deflections(rest_full, task_full, absolute=absolute)

    rest_dec = data_dict_centiles[rest_scan]
    try:
        task_dec = data_dict_centiles[task_scan]
    except (KeyError, TypeError):
        task_dec = data_dict_centiles[rest_scan]
    deflection_dec = compute_deflections(rest_dec, task_dec, absolute=absolute)
    n_deciles = rest_dec.shape[1]

    # ===== Fit GAMs (decile + region). lam is a float (fixed) or 'gcv' (per-series grid search). =====
    results_dec = fit_all_panels(age, rest_dec, task_dec, deflection_dec, age_grid,
                                 n_splines=n_splines, lam=lam, demean=True, show_ci=False,
                                 nuisance=nuisance)

    apply_tsnr_mask = False  # set True to exclude low-tSNR regions
    if apply_tsnr_mask:
        region_mask = (df_tsnr[task_scan] < np.quantile(df_tsnr[task_scan], q=0.1)).values
    else:
        region_mask = None
    rest_reg, task_reg, defl_reg, sa_axis_reg = apply_region_mask(
        region_mask, rest_full, task_full, deflection_full, sa_axis=sa_axis)
    results_reg = fit_all_panels(age, rest_reg, task_reg, defl_reg, age_grid,
                                 n_splines=n_splines, lam=lam, demean=True, show_ci=False,
                                 nuisance=nuisance)
    print(f'GAMs fit ({fit_recipe}). Decile {deflection_dec.shape} | Regional {defl_reg.shape} | '
          f'Nuisance {None if nuisance is None else nuisance.shape}')
    if lam == 'gcv':
        _ed = np.array([g.statistics_['edof'] for g in results_reg['deflection']['gams'] if g is not None])
        _lam_med = np.nanmedian(results_reg['deflection']['lams'])
        print(f'  GCV lam median {_lam_med:.2f} | regional edof median {np.median(_ed):.2f} '
              f'(incl. nuisance terms)')

    # ===== Bootstrap: each subject-resample refit is reduced ONCE to the derivative × S-A axis
    # correlation curve (this figure's CI), the stage-1/stage-2 net-change maps (exported for
    # Figure 5's Yeo-7 enrichment, so fig5 does zero GAM fitting), AND a data-driven stage boundary
    # detected per resample from stage_boundary_signal (maps integrate over each resample's own
    # window). boundary_report_signals are detected too, for robustness CIs. Parallel over resamples. =====
    _boundary_signals = (stage_boundary_signal,) + tuple(boundary_report_signals)
    # Freeze the per-region smoothing from the full-sample deflection fit (the GCV-selected lam, or
    # the fixed scalar) so every resample reuses the same smoothing rather than re-selecting it.
    frozen_lam = results_reg['deflection']['lams']
    boot = bootstrap_gam_maps(
        defl_reg, age, age_grid, sa_axis_vals=sa_axis_reg, inflection_age=stage_boundary_signal,
        report_boundary_signals=boundary_report_signals,
        n_splines=n_splines, lam=frozen_lam, demean=True, nuisance=nuisance, annualized=True,
        n_boot=n_boot, method='pearson', random_state=boot_seed, n_jobs=N_JOBS)
    boot_curves = boot['sa_curves']
    boot_stage1, boot_stage2 = boot['stage1'], boot['stage2']
    boot_boundary = {s: boot[f'boundary_{s}'] for s in _boundary_signals}
    r_ci_lo = np.nanpercentile(boot_curves, 2.5,  axis=0)
    r_ci_hi = np.nanpercentile(boot_curves, 97.5, axis=0)

    # Point-estimate boundary on the full-sample regional fit: defines the stage boundary fig5 uses
    # for its observed maps and annotates this figure; the bootstrap distribution above gives its CI.
    boundary_pt    = {s: detect_change_point(results_reg['deflection']['pred'], age_grid, signal=s)
                      for s in _boundary_signals}
    inflection_age = float(boundary_pt[stage_boundary_signal]['location'])
    # Multi-change-point boundaries for the figure annotation (draws n_bkps vertical lines); n_bkps=1
    # reproduces the single stage boundary. Detected on the same regional signal that drives the maps.
    boundary_ages  = detect_change_point(results_reg['deflection']['pred'], age_grid,
                                           signal=stage_boundary_signal, n_bkps=n_bkps)['locations']

    print(f'Bootstrap done ({n_boot} iters, n_jobs={N_JOBS}). 95% CI at age midpoint: '
          f'[{r_ci_lo[100]:.3f}, {r_ci_hi[100]:.3f}]')
    _b = boot_boundary[stage_boundary_signal]
    print(f'Data-driven stage boundary [{stage_boundary_signal}]: {inflection_age:.2f} yr '
          f'(point est.), 95% CI [{np.nanpercentile(_b, 2.5):.2f}, {np.nanpercentile(_b, 97.5):.2f}] yr.')
    if n_bkps > 1:
        print(f'  n_bkps={n_bkps} boundaries (annotation only): {[round(a, 2) for a in boundary_ages]} yr')
    for s in boundary_report_signals:
        _bs = boot_boundary[s]
        print(f'  boundary [{s}] (robustness): {boundary_pt[s]["location"]:.2f} yr (point est.), '
              f'95% CI [{np.nanpercentile(_bs, 2.5):.2f}, {np.nanpercentile(_bs, 97.5):.2f}] yr.')
    print(f'Bootstrap stage maps for fig5: stage1 {boot_stage1.shape}, stage2 {boot_stage2.shape}.')

    # ===== Assemble outputs (figure inputs + cross-notebook export for fig5) =====
    save_dict = {
        'age_grid':    age_grid,
        'sa_axis_reg': sa_axis_reg,
        'task_scan':   task_scan,
        'n_deciles':   n_deciles,
    }
    for sig in ('rest', 'task', 'deflection'):
        save_dict[f'{sig}_deriv']     = results_reg[sig]['deriv']
        save_dict[f'{sig}_pred']      = results_reg[sig]['pred']
        save_dict[f'{sig}_deriv_dec'] = results_dec[sig]['deriv']
        save_dict[f'{sig}_pred_dec']  = results_dec[sig]['pred']
        save_dict[f'{sig}_partial_r2'] = partial_r2_smooth(
            {'rest': rest_reg, 'task': task_reg, 'deflection': defl_reg}[sig],
            age, results_reg[sig]['gams'], nuisance=nuisance, demean=True
        )
    save_dict['r_vals']      = compute_deriv_sa_corr(
        results_reg['deflection']['deriv'], sa_axis_reg, method='pearson')
    save_dict['boot_curves']    = boot_curves
    save_dict['boot_stage1']    = boot_stage1     # (n_boot, n_regions) fig5 enrichment bootstrap
    save_dict['boot_stage2']    = boot_stage2
    save_dict['inflection_age'] = inflection_age  # data-driven boundary point estimate (yr); fig5 reads this
    save_dict['stage_boundary_signal'] = stage_boundary_signal
    save_dict['boundary_ages']  = boundary_ages   # n_bkps annotation boundaries (yr)
    save_dict['n_bkps']         = n_bkps
    for s in _boundary_signals:
        save_dict[f'boundary_{s}']    = boot_boundary[s]                       # (n_boot,) boundary CI
        save_dict[f'boundary_pt_{s}'] = float(boundary_pt[s]['location'])  # full-sample point est.
    save_dict['n_boot']         = n_boot
    save_dict['boot_seed']      = boot_seed
    save_dict['fit_recipe']     = fit_recipe                                   # provenance
    save_dict['n_splines']      = n_splines
    save_dict['gam_lam']        = lam if isinstance(lam, str) else float(lam)
    save_dict['frozen_lam']     = frozen_lam                                   # (n_regions,) deflection smoothing used
    save_dict['r_ci_lo']     = r_ci_lo
    save_dict['r_ci_hi']     = r_ci_hi
    save_dict['rest_reg']    = rest_reg
    save_dict['defl_reg']    = defl_reg
    save_dict['sa_axis']     = sa_axis
    save_dict['df_behavior'] = df_behavior

    np.save(gam_cache_file, save_dict)
    print(f'Computed and saved GAM + bootstrap outputs -> {gam_cache_file}')
    for sig in ('rest', 'task', 'deflection'):
        r2 = save_dict[f'{sig}_partial_r2']
        print(f"  {sig}_partial_r2: shape={r2.shape}, mean={np.nanmean(r2):.4f}, max={np.nanmax(r2):.4f}")

In [ ]:
# Reconstruct the figure inputs from save_dict (identical whether loaded or freshly computed).
age_grid    = save_dict['age_grid']
sa_axis_reg = save_dict['sa_axis_reg']
n_deciles   = save_dict['n_deciles']
r_ci_lo     = save_dict['r_ci_lo']
r_ci_hi     = save_dict['r_ci_hi']

# Data-driven stage boundary (point estimate + subject-bootstrap 95% CI) for annotation
boundary_signal = save_dict.get('stage_boundary_signal', 'deriv1')
boundary_pt_age = save_dict.get('inflection_age')                  # float, data-driven point estimate (yr)
_bd             = save_dict.get(f'boundary_{boundary_signal}')     # (n_boot,) or None
boundary_ci     = np.nanpercentile(_bd, [2.5, 97.5]) if _bd is not None else None
# n_bkps change-point boundaries drawn as vertical lines (falls back to the single boundary)
boundary_ages   = save_dict.get('boundary_ages',
                                [boundary_pt_age] if boundary_pt_age is not None else [])

results_reg = {sig: {'deriv': save_dict[f'{sig}_deriv'],     'pred': save_dict[f'{sig}_pred']}
               for sig in ('rest', 'task', 'deflection')}
results_dec = {sig: {'deriv': save_dict[f'{sig}_deriv_dec'], 'pred': save_dict[f'{sig}_pred_dec']}
               for sig in ('rest', 'task', 'deflection')}

cmap          = register_custom_colormaps()['sydnor_sa']   # registers all custom maps; sydnor_sa colours the decile lines + S-A colorbar
series_colors = get_series_colors('decile', n_deciles, cmap)
draw_order    = get_draw_order('decile', n_deciles)

# Figure 4

In [ ]:
# --- Parameters ---
signal_type = 'deflection'  # 'rest', 'task', or 'deflection'
plot_level  = 'region'      # 'region' or 'decile'
heatmap_cmap = 'blue_pink'  # diverging map for panel A (blue<->red; avoids the orange/purple of the S-A axis). preview all via show_colormaps()
# ------------------

_signal_labels = {'rest': 'REST INTs', 'task': 'Task INTs', 'deflection': 'INT Deflections'}
lw = 1.8

# Heatmap data
_reg_deriv = results_reg[signal_type]['deriv']

if plot_level == 'region':
    sort_idx     = np.argsort(sa_axis_reg)
    deriv_sorted = _reg_deriv[sort_idx]
    ylabel       = 'S-A axis'
    bottom_label = 'Sensorimotor pole'
    top_label    = 'Association pole'
else:
    deriv_sorted = results_dec[signal_type]['deriv']
    ylabel       = 'S-A axis decile'
    bottom_label = 'Sensorimotor'
    top_label    = 'Association'

n_rows = deriv_sorted.shape[0]
vmax   = np.nanpercentile(np.abs(deriv_sorted), 98)

r_vals = compute_deriv_sa_corr(_reg_deriv, sa_axis_reg, method='pearson')

# SA-decile 2nd derivative (acceleration) of the fit: central differences of pred with
# edge_order=2 (matches build_signal). edge_order=2 avoids the compounded boundary
# artifact a plain double np.gradient produces at the age extremes.
_pred_dec   = results_dec[signal_type]['pred']
_deriv2_dec = np.gradient(np.gradient(_pred_dec, age_grid, axis=1, edge_order=2),
                          age_grid, axis=1, edge_order=2)

# Data-driven change-point boundaries: draw one dashed vertical line per detected boundary
# (n_bkps of them; a single line when n_bkps=1) on every panel.
def _draw_stage_boundary(ax, on_image=False):
    _z = None if on_image else 0        # behind the data curves on the line plots
    for _a in boundary_ages:
        ax.axvline(_a, color='0.2', linestyle='--', lw=0.9, alpha=0.9,
                   **({} if _z is None else {'zorder': _z + 0.5}))

# Bold panel label (A, B, ...) at each subplot's top-left corner.
def _panel_label(ax, letter):
    ax.annotate(letter, xy=(0.0, 1.0), xycoords='axes fraction', xytext=(-22, 6),
                textcoords='offset points', fontsize=11, fontweight='bold',
                va='bottom', ha='left', annotation_clip=False)

# Figure layout
fig      = plt.figure(figsize=(8.5, 6.5), dpi=300)
gs_outer = GridSpec(1, 2, figure=fig, width_ratios=[1, 1], wspace=0.15)

# Left column: heatmap | narrow vertical colorbar
gs_left   = GridSpecFromSubplotSpec(1, 2, subplot_spec=gs_outer[0],
                                    width_ratios=[1, 0.025], wspace=0.05)
ax_heat   = fig.add_subplot(gs_left[0])
ax_cbar_h = fig.add_subplot(gs_left[1])

# Right column: pred / deriv1 / deriv2 / Pearson r (4 panels; SA-decile colorbar added as overlay)
gs_right = GridSpecFromSubplotSpec(4, 1, subplot_spec=gs_outer[1],
                                   height_ratios=[1, 1, 1, 1], hspace=0.3)
ax0 = fig.add_subplot(gs_right[0])
ax1 = fig.add_subplot(gs_right[1], sharex=ax0)
ax2 = fig.add_subplot(gs_right[2], sharex=ax0)
ax3 = fig.add_subplot(gs_right[3], sharex=ax0)

# Heatmap
im = ax_heat.imshow(
    deriv_sorted, aspect='auto', origin='lower',
    extent=[age_grid[0], age_grid[-1], 0, n_rows],
    cmap=heatmap_cmap, vmin=-vmax, vmax=vmax, interpolation='nearest',
)
_draw_stage_boundary(ax_heat, on_image=True)

ax_heat.set_xlabel('Age (years)', fontsize=8, labelpad=2)
ax_heat.set_ylabel(ylabel, fontsize=8, labelpad=4)
if plot_level == 'region':
    ax_heat.set_yticks([0, n_rows // 2, n_rows])
    ax_heat.set_yticklabels(['', '', ''], fontsize=8)
else:
    ax_heat.set_yticks(np.arange(n_rows) + 0.5)
    ax_heat.set_yticklabels([str(i + 1) for i in range(n_rows)], fontsize=8)
ax_heat.tick_params(labelsize=8, pad=1)

ax_heat.annotate(top_label,    xy=(age_grid[0], n_rows), xytext=(-8, -1.5), rotation=90,
                 textcoords='offset points', fontsize=8, va='top',    annotation_clip=False)
ax_heat.annotate(bottom_label, xy=(age_grid[0], 0),      xytext=(-8,  1.5), rotation=90,
                 textcoords='offset points', fontsize=8, va='bottom', annotation_clip=False)
ax_heat.annotate('', xy=(age_grid[-1], n_rows + n_rows * 0.015),
                 xytext=(age_grid[0],  n_rows + n_rows * 0.015),
                 arrowprops=dict(arrowstyle='->', color='k', lw=0.8), annotation_clip=False)
ax_heat.text((age_grid[0] + age_grid[-1]) / 2, n_rows + n_rows * 0.025,
             'Age-resolved analysis', ha='center', va='bottom', fontsize=8, clip_on=False)

cb_h = fig.colorbar(im, cax=ax_cbar_h, orientation='vertical')
cb_h.set_label(f'{_signal_labels[signal_type]}\n(1st derivative)', fontsize=8)
cb_h.ax.yaxis.set_label_position('right')
cb_h.ax.tick_params(labelsize=6)

# Line plots: pred (ax0), 1st derivative (ax1), 2nd derivative (ax2)
for i in draw_order:
    ax0.plot(age_grid, results_dec[signal_type]['pred'][i], color=series_colors[i], lw=lw)
style_ax(ax0, 'Age (years)', f'{_signal_labels[signal_type]}\n(prediction)', None, demean=True)
ax0.set_xlabel('')
ax0.set_box_aspect(0.8)
plt.setp(ax0.get_xticklabels(), visible=False)

for i in draw_order:
    ax1.plot(age_grid, results_dec[signal_type]['deriv'][i], color=series_colors[i], lw=lw)
style_ax(ax1, 'Age (years)', f'{_signal_labels[signal_type]}\n(1st derivative)', None, demean=True)
ax1.set_xlabel('')
ax1.set_box_aspect(0.8)
plt.setp(ax1.get_xticklabels(), visible=False)

for i in draw_order:
    ax2.plot(age_grid, _deriv2_dec[i], color=series_colors[i], lw=lw)
style_ax(ax2, 'Age (years)', f'{_signal_labels[signal_type]}\n(2nd derivative)', None, demean=True)
ax2.set_xlabel('')
ax2.set_box_aspect(0.8)
plt.setp(ax2.get_xticklabels(), visible=False)

# r(SA axis × regional GAM derivative) with bootstrap 95% CI and annotations
plot_deriv_sa_corr_with_ci(ax3, age_grid, r_vals, r_ci_lo, r_ci_hi, lw=lw)
ax3.set_xlabel('Age (years)', fontsize=8, labelpad=1)
ax3.set_ylabel(f'Pearson $r$\n(1st deriv. × S-A axis)', fontsize=8, labelpad=1)
ax3.tick_params(labelsize=8, pad=0.25)
ax3.set_box_aspect(0.8)
ax3.spines[['top', 'right']].set_visible(False)

# tau line(s) on the right-column panels (matching the heatmap)
for _ax in (ax0, ax1, ax2, ax3):
    _draw_stage_boundary(_ax)

# Bold panel labels: A = heatmap, B-E = right column (top -> bottom)
for _ax, _lab in [(ax_heat, 'A'), (ax0, 'B'), (ax1, 'C'), (ax2, 'D'), (ax3, 'E')]:
    _panel_label(_ax, _lab)

# SA-decile colorbar on the right-hand side of the top-right panel (ax0)
fig.canvas.draw()
add_sa_decile_colorbar(fig, ax0, cmap, n_deciles, orientation='vertical')

fig.savefig(os.path.join(nas_scratch_dir, f'fig4_combined_{task_scan}.png'),
            dpi=1000, bbox_inches='tight', transparent=False)
plt.show()

# Figure 4 — supplement: per-decile GAM fit check

In [ ]:
# ── Supplementary: per-decile GAM fit sanity check ──────────────────────────
# For a few S-A deciles, overlay the GAM prediction on the raw (demeaned) INT-deflection data it was
# fit to. The raw decile deflections are recomputed here (mirroring the fit exactly), so this works
# whether the GAM cache was loaded or freshly computed. The curve is the age smooth evaluated at mean
# nuisance and demeaned, so the raw data is demeaned to match; extra scatter vs the curve is the
# nuisance covariates (sex, meanFD, mean rest INT) + noise, not lack of fit.
import matplotlib.patheffects as pe

supp_deciles = [1, 10]   # 1-indexed S-A deciles to show (sensorimotor -> association)

_rest_dec = data_dict_centiles[rest_scan]
try:
    _task_dec = data_dict_centiles[task_scan]
except (KeyError, TypeError):
    _task_dec = data_dict_centiles[rest_scan]
deflection_dec_raw = compute_deflections(_rest_dec, _task_dec, absolute=absolute)   # (n_subjects, n_deciles)
age_subj = df_behavior['age'].values
pred_dec = results_dec['deflection']['pred']                                        # (n_deciles, n_grid), demeaned

fig, axes = plt.subplots(1, len(supp_deciles), figsize=(2 * len(supp_deciles), 2.25),
                         dpi=300, sharex=True)
axes = np.atleast_1d(axes)
for ax, d in zip(axes, supp_deciles):
    j     = d - 1
    y_raw = deflection_dec_raw[:, j].astype(float)
    y_raw = y_raw - np.nanmean(y_raw)                        # demean to match the fitted curve
    ax.scatter(age_subj, y_raw, s=8, color='0.6', alpha=0.4, edgecolors='none', zorder=1)
    # Curve coloured by S-A decile (series_colors), with a dark outline via path_effects so the
    # near-white mid-deciles stay visible against the white background.
    ax.plot(age_grid, pred_dec[j], color=series_colors[j], lw=2.4, zorder=3,
            path_effects=[pe.Stroke(linewidth=3.4, foreground='0.25'), pe.Normal()])
    ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.4, zorder=2)
    lo, hi = np.nanpercentile(y_raw, [1, 99])               # focus the view on the bulk (robust to outliers)
    ax.set_ylim(lo - 0.15 * (hi - lo), hi + 0.15 * (hi - lo))
    ax.set_title(f'S-A decile {d}', fontsize=8)
    ax.set_xlabel('Age (years)', fontsize=8, labelpad=2)
    ax.tick_params(labelsize=8, pad=1)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('INT Deflections', fontsize=8)
fig.suptitle(f'Per-decile GAM fit ({fit_recipe}, n_splines={n_splines})', fontsize=8, y=1.02)

fig.tight_layout()
fig.savefig(os.path.join(nas_scratch_dir, f'fig4_supp_decile_fits_{task_scan}.png'),
            dpi=300, bbox_inches='tight', transparent=False)
plt.show()